# Exercícios
## 1) Modificar a profundidade da árvore e comparar resultados
**Objetivo:**

Entender como a profundidade (max_depth) influencia o underfitting e o overfitting.

Altere o parâmetro da árvore de decisão:

model = DecisionTreeClassifier(max_depth=3)

E teste diferentes valores:

max_depth = 1;
max_depth = 5;
max_depth = None (profundidade ilimitada)

Para cada profundidade:

1. Treine o modelo.
2. Plote a árvore de decisão.
3. Gere a fronteira de decisão.
4. Calcule as métricas: accuracy, precisão, recall, F1.

**Pergunta final:** Como a profundidade influencia o overfitting e o desempenho no teste?

## 2) Alterar o dataset e reavaliar o modelo
**Objetivo:**

Explorar como diferentes distribuições de dados afetam o comportamento da árvore.

Modifique o bloco:

X, y = make_classification(...)

Testando diferentes configurações:

1. Aumente o ruído dos rótulos
flip_y=0.15
2. Aumente o número de clusters por classe
n_clusters_per_class=2
3. Aumente o número de features informativas
n_features=4
n_informative=4

Para cada cenário:

1. Refaça o treino.
2. Plote (quando possível).
3. Gere a matriz de confusão.
4. Calcule accuracy, precisão, recall e F1.

**Pergunta final:** Em quais condições o modelo piora mais? Por quê?

In [ ]:
#Bibliotecas principais para trabalhar com dados, gráficos e árvore de decisão
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Geração do dataset, divisão treino/teste e modelo de árvore
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree

# Métricas usadas para avaliar o desempenho do modelo
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

In [ ]:
#Função simples para deixar o nome da profundidade mais bonito nos títulos e tabelas
def nome_profundidade(depth):
    return "None" if depth is None else str(depth)


#Centralizei as métricas em uma função para não repetir o mesmo bloco várias vezes
def calcular_metricas(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        #zero_division=0 evita erro caso o modelo não preveja alguma classe
        "precisão": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1-score": f1_score(y_true, y_pred, zero_division=0)
    }


#Esta função só funciona bem quando temos duas variáveis de entrada: x1 e x2
def plotar_fronteira(model, X, y, titulo):
    """Plota a fronteira de decisão apenas para datasets com 2 features."""
    if X.shape[1] != 2:
        print(f"Fronteira de decisão não plotada para '{titulo}', pois X possui {X.shape[1]} features.")
        return

    #Cria uma área um pouco maior que os pontos reais para desenhar a fronteira
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1

    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 300),
        np.linspace(y_min, y_max, 300)
    )

    #Cada ponto do grid será classificado pelo modelo para formar as regiões coloridas
    grid_points = np.c_[xx.ravel(), yy.ravel()]
    Z = model.predict(grid_points).reshape(xx.shape)

    plt.figure(figsize=(8, 6))
    plt.contourf(xx, yy, Z, alpha=0.3, cmap="coolwarm")
    plt.scatter(X[:, 0], X[:, 1], c=y, edgecolor="k", cmap="coolwarm", s=35)
    plt.title(titulo, fontsize=14, fontweight="bold")
    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.tight_layout()
    plt.show()


#Avalia tanto no treino quanto no teste, pois isso ajuda a perceber underfitting e overfitting
def avaliar_modelo(model, X_train, X_test, y_train, y_test):
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    metricas_train = calcular_metricas(y_train, y_pred_train)
    metricas_test = calcular_metricas(y_test, y_pred_test)

    return metricas_train, metricas_test, y_pred_test

## 1) Modificar a profundidade da árvore e comparar resultados
Criação de dataset base com 2 features para permitir o desenho da fronteira de decisão.

In [ ]:
#Dataset base com apenas 2 features para conseguirmos visualizar a fronteira de decisão em 2D
X, y = make_classification(
    n_samples=500,
    n_features=2,
    n_informative=2,
    n_redundant=0,
    n_repeated=0,
    n_clusters_per_class=1,
    class_sep=1.2,
    flip_y=0.03,
    random_state=42  #mantém o mesmo dataset sempre que o código for executado
)

#Separação entre treino e teste
#stratify=y mantém a proporção das classes parecida nos dois conjuntos
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

feature_names = ["x1", "x2"]
class_names = ["0", "1"]

In [ ]:
#Profundidades pedidas no exercício, incluindo None para deixar a árvore crescer sem limite
profundidades = [1, 3, 5, None]

modelos = []
resultados = []

for depth in profundidades:
    #Cria uma árvore nova para cada profundidade
    model = DecisionTreeClassifier(max_depth=depth, random_state=42)
    model.fit(X_train, y_train)
    modelos.append(model)

    #Avaliamos treino e teste para comparar generalização
    metricas_train, metricas_test, y_pred_test = avaliar_modelo(
        model, X_train, X_test, y_train, y_test
    )

    #Guardo os resultados em lista para transformar em tabela no final
    resultados.append({
        "max_depth": nome_profundidade(depth),
        "accuracy treino": metricas_train["accuracy"],
        "accuracy teste": metricas_test["accuracy"],
        "precisão teste": metricas_test["precisão"],
        "recall teste": metricas_test["recall"],
        "f1-score teste": metricas_test["f1-score"]
    })

resultados_depth = pd.DataFrame(resultados)
resultados_depth

In [ ]:
#Visualização das quatro árvores lado a lado para comparar a complexidade de cada uma
fig, axes = plt.subplots(2, 2, figsize=(24, 16))
fig.suptitle("Comparação das Árvores por Profundidade", fontsize=22, fontweight="bold")

for ax, model, depth in zip(axes.flatten(), modelos, profundidades):
    plot_tree(
        model,
        filled=True,
        feature_names=feature_names,
        class_names=class_names,
        ax=ax,
        fontsize=8
    )
    ax.set_title(f"Decision Tree — max_depth={nome_profundidade(depth)}", fontsize=14)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

In [ ]:
#Para desenhar a fronteira, criamos vários pontos artificiais cobrindo o plano x1 x x2
xx, yy = np.meshgrid(
    np.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 300),
    np.linspace(X[:, 1].min() - 1, X[:, 1].max() + 1, 300)
)
grid_points = np.c_[xx.ravel(), yy.ravel()]

fig, axes = plt.subplots(2, 2, figsize=(18, 14))
fig.suptitle("Fronteiras de Decisão por Profundidade", fontsize=22, fontweight="bold")

for ax, model, depth in zip(axes.flatten(), modelos, profundidades):
    #O modelo prevê a classe de cada ponto do grid, isso forma as regiões coloridas
    Z = model.predict(grid_points).reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.3, cmap="coolwarm")
    ax.scatter(X[:, 0], X[:, 1], c=y, edgecolor="k", cmap="coolwarm", s=30)

    ax.set_title(f"Decision Boundary — max_depth={nome_profundidade(depth)}", fontsize=14)
    ax.set_xlabel("x1")
    ax.set_ylabel("x2")

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

In [ ]:
#Matriz de confusão para ver onde o modelo acertou e errou no conjunto de teste
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle("Matrizes de Confusão por Profundidade", fontsize=18, fontweight="bold")

for ax, model, depth in zip(axes.flatten(), modelos, profundidades):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)

    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(f"max_depth={nome_profundidade(depth)}")

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

### Resposta - Exercício 1

A profundidade controla a complexidade da árvore:
- `max_depth=1`: árvore muito simples. Geralmente sofre **underfitting**, pois aprende poucas regras e pode errar tanto no treino quanto no teste.
- `max_depth=3`: costuma ser mais equilibrada, pois aprende padrões importantes sem criar muitas divisões.
- `max_depth=5`: aprende fronteiras mais detalhadas. Pode melhorar o teste, mas já começa a ter risco de overfitting dependendo do dataset.
- `max_depth=None`: a árvore cresce sem limite. Normalmente aumenta muito o desempenho no treino, mas pode piorar no teste porque memoriza ruídos e particularidades dos dados de treino.
Portanto, quanto maior a profundidade, maior a complexidade do modelo. Isso pode melhorar o desempenho até certo ponto, mas profundidades muito grandes tendem ao **overfitting**.

## 2) Alterar o dataset e reavaliar o modelo

Profundidade fixa (`max_depth=5`) para comparar como mudanças no dataset afetam o desempenho.

In [ ]:
#Cada cenário altera uma característica do dataset para observar como a árvore se comporta
cenarios = {
    "Base": {
        "n_samples": 500,
        "n_features": 2,
        "n_informative": 2,
        "n_redundant": 0,
        "n_repeated": 0,
        "n_clusters_per_class": 1,
        "class_sep": 1.2,
        "flip_y": 0.03,
        "random_state": 42
    },
    "Mais ruído — flip_y=0.15": {
        "n_samples": 500,
        "n_features": 2,
        "n_informative": 2,
        "n_redundant": 0,
        "n_repeated": 0,
        "n_clusters_per_class": 1,
        "class_sep": 1.2,
        "flip_y": 0.15,  #aumenta a troca aleatória dos rótulos, deixando os dados mais confusos
        "random_state": 42
    },
    "Mais clusters — n_clusters_per_class=2": {
        "n_samples": 500,
        "n_features": 2,
        "n_informative": 2,
        "n_redundant": 0,
        "n_repeated": 0,
        "n_clusters_per_class": 2,  #cada classe fica dividida em mais grupos
        "class_sep": 1.2,
        "flip_y": 0.03,
        "random_state": 42
    },
    "Mais features — n_features=4, n_informative=4": {
        "n_samples": 500,
        "n_features": 4,
        "n_informative": 4,  #agora as quatro variáveis carregam informação útil para o modelo
        "n_redundant": 0,
        "n_repeated": 0,
        "n_clusters_per_class": 1,
        "class_sep": 1.2,
        "flip_y": 0.03,
        "random_state": 42
    }
}

In [ ]:
#Aqui automatizamos o treino e a avaliação para cada cenário do exercício 2
resultados_cenarios = []
modelos_cenarios = {}

def executar_cenario(nome, params, max_depth=5):
    print("=" * 70)
    print(f"CENÁRIO: {nome}")
    print("=" * 70)

    #Cria o dataset de acordo com os parâmetros do cenário atual
    X_c, y_c = make_classification(**params)

    #Mantém a mesma lógica de treino/teste usada no exercício 1
    X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
        X_c, y_c,
        test_size=0.30,
        random_state=42,
        stratify=y_c
    )

    #Usei max_depth=5 como profundidade fixa para comparar apenas o efeito do dataset
    model_c = DecisionTreeClassifier(max_depth=max_depth, random_state=42)
    model_c.fit(X_train_c, y_train_c)

    y_pred_c = model_c.predict(X_test_c)
    metricas = calcular_metricas(y_test_c, y_pred_c)

    #Salva os resultados para montar uma tabela comparativa no final
    resultados_cenarios.append({
        "cenário": nome,
        "n_features": params["n_features"],
        "flip_y": params["flip_y"],
        "n_clusters_per_class": params["n_clusters_per_class"],
        "accuracy": metricas["accuracy"],
        "precisão": metricas["precisão"],
        "recall": metricas["recall"],
        "f1-score": metricas["f1-score"]
    })

    #Guardo os objetos principais caso seja necessário consultar depois
    modelos_cenarios[nome] = {
        "model": model_c,
        "X": X_c,
        "y": y_c,
        "X_test": X_test_c,
        "y_test": y_test_c,
        "y_pred": y_pred_c,
        "feature_names": [f"x{i+1}" for i in range(params["n_features"])]
    }

    print("MÉTRICAS DO MODELO")
    print(f"Accuracy : {metricas['accuracy']:.3f}")
    print(f"Precisão : {metricas['precisão']:.3f}")
    print(f"Recall   : {metricas['recall']:.3f}")
    print(f"F1-score : {metricas['f1-score']:.3f}")

    #Matriz de confusão do cenário atual
    cm = confusion_matrix(y_test_c, y_pred_c)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["0", "1"])
    disp.plot(cmap="Blues")
    plt.title(f"Matriz de Confusão — {nome}")
    plt.tight_layout()
    plt.show()

    #Plot da árvore treinada nesse cenário
    plt.figure(figsize=(22, 10))
    plot_tree(
        model_c,
        filled=True,
        feature_names=[f"x{i+1}" for i in range(params["n_features"])],
        class_names=["0", "1"],
        fontsize=8
    )
    plt.title(f"Árvore de Decisão — {nome}", fontsize=16, fontweight="bold")
    plt.tight_layout()
    plt.show()

    #A fronteira só é plotada quando existem exatamente 2 features
    if params["n_features"] == 2:
        plotar_fronteira(model_c, X_c, y_c, f"Fronteira de Decisão — {nome}")
    else:
        print("Fronteira de decisão não gerada porque este cenário possui 4 features.")


for nome, params in cenarios.items():
    executar_cenario(nome, params, max_depth=5)

resultados_cenarios_df = pd.DataFrame(resultados_cenarios)
resultados_cenarios_df

In [ ]:
#Ordena os cenários pelo F1-score para facilitar a comparação final
#Quanto maior o F1-score, melhor o equilíbrio entre precisão e recall
resultados_cenarios_df.sort_values(by="f1-score", ascending=False)

### Resposta Exercício 2

O modelo tende a piorar mais quando os dados ficam mais difíceis de separar.
Com `flip_y=0.15` costuma prejudicar bastante porque altera artificialmente parte dos rótulos, isso cria exemplos contraditórios como pontos parecidos podem aparecer com classes diferentes. A árvore pode tentar memorizar esses ruídos, aumentando o risco de overfitting.

Com `n_clusters_per_class=2` também pode piorar porque cada classe passa a ter mais de uma região no espaço, a fronteira de decisão precisa ficar mais fragmentada para separar corretamente os grupos, aumentando a complexidade do problema.

Com `n_features=4` e `n_informative=4` não é necessariamente pior, como há mais variáveis informativas, o modelo pode ganhar mais informação para separar as classes. Porém, a interpretação visual fica mais difícil, porque não conseguimos representar a fronteira completa em apenas duas dimensões.

Em geral, a árvore piora mais quando há **ruído nos rótulos** e quando as classes ficam **mais espalhadas ou fragmentadas**, pois isso força o modelo a criar regras mais específicas e aumenta o risco de overfitting.